In [124]:
from Bio import SeqIO
from pathlib import Path
from collections import Counter, defaultdict

In [96]:
seq_records = SeqIO.parse(handle=Path("./data/dna2.fasta"), format="fasta")

In [48]:
sequence_lengths = {}

for record in SeqIO.parse(handle="./data/dna2.fasta", format="fasta"):
    length = len(record.seq)    #get sequence length
    sequence_lengths[record.id] = length

In [51]:
sequence_lengths

{'gi|142022655|gb|EQ086233.1|91': 4635,
 'gi|142022655|gb|EQ086233.1|304': 1151,
 'gi|142022655|gb|EQ086233.1|255': 4894,
 'gi|142022655|gb|EQ086233.1|45': 3511,
 'gi|142022655|gb|EQ086233.1|396': 4076,
 'gi|142022655|gb|EQ086233.1|250': 2867,
 'gi|142022655|gb|EQ086233.1|322': 442,
 'gi|142022655|gb|EQ086233.1|88': 890,
 'gi|142022655|gb|EQ086233.1|594': 967,
 'gi|142022655|gb|EQ086233.1|293': 4338,
 'gi|142022655|gb|EQ086233.1|75': 1352,
 'gi|142022655|gb|EQ086233.1|454': 4564,
 'gi|142022655|gb|EQ086233.1|16': 4804,
 'gi|142022655|gb|EQ086233.1|584': 964,
 'gi|142022655|gb|EQ086233.1|4': 2095,
 'gi|142022655|gb|EQ086233.1|277': 1432,
 'gi|142022655|gb|EQ086233.1|346': 115,
 'gi|142022655|gb|EQ086233.1|527': 2646}

In [53]:
print(f"There are {len(sequence_lengths)} records in the file")

There are 18 records in the file


In [56]:
longest_sequence = max(list(sequence_lengths.values()))
print(f"The longest sequence is {longest_sequence} bp")

The longest sequence is 4894 bp


In [58]:
shortest_sequence = min(list(sequence_lengths.values()))
print(f"The shortest sequence is {shortest_sequence} bp")

The shortest sequence is 115 bp


In [102]:
STOP_CODONS = {"TAA", "TAG", "TGA"}

In [103]:
def find_orfs_fasta(fasta_file: str | Path, frame: int=1) -> list[dict]:
    """
    Find all forward-strand ORFs in a given reading frame.

    frame: 1, 2, or 3
    returns:
        all_orfs: list of ORF dictionaries
        longest_in_file: dictionary for longest ORF overall
    """
    if frame not in {1, 2, 3}:
        raise ValueError("frame must be 1, 2, or 3")

    offset = frame - 1
    all_orfs = []

    for record in SeqIO.parse(handle=fasta_file, format="fasta"):
        seq_id = record.id
        seq = str(record.seq).upper()

        i = offset
        while i <= len(seq) - 3:
            codon = seq[i:i+3]

            if codon == "ATG":
                j = i + 3

                while j <= len(seq) - 3:
                    stop = seq[j:j+3]

                    if stop in STOP_CODONS:
                        orf_seq = seq[i:j+3]

                        all_orfs.append({
                            "sequence_id": seq_id,
                            "start": i + 1,      # 1-based position
                            "end": j + 3,
                            "length": len(orf_seq),
                            "orf": orf_seq,
                            "frame": frame
                        })
                        break

                    j += 3

            i += 3

    return all_orfs

In [104]:
all_orfs_2 = find_orfs_fasta(fasta_file="./data/dna2.fasta", frame=2)
longest_in_file = max(all_orfs_2, key=lambda x: x["length"], default=None)

longest_in_file


{'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
 'start': 3071,
 'end': 4528,
 'length': 1458,
 'orf': 'ATGGCAATCCTGATTCGTGGCGGCACCGTGGTCGATGCGGACCGTTCCTACCGCGCGGACGTGCTCTGCGCAGCCCCGGAGGACGGCGGCACGATCCTGCAGATCGCCGGGCAGATCGATGCGCCGGCCGGCGCGACCGTCGTCGATGCGCACGACCAGTACGTGATGCCGGGCGGCATCGATCCGCATACGCACATGGAACTGCCGTTCATGGGCACGACCGCGAGCGACGATTTCTACTCGGGTACGGCCGCCGGGCTCGCGGGCGGCACGACGAGCATCATCGACTTCGTGATCCCGAGCCCGAAGCAGCCGCTGATGGACGCGTTCCATGCCTGGCGCGGCTGGGCCGAGAAGGCGGCGGCCGACTACGGCTTCCACGTGGCCGTGACGTGGTGGGACGAGAGTGTGCACCGCGACATGGGCACGCTCGTGCGCGAACACGGCGTGTCGAGCTTCAAGCACTTCATGGCGTACAAGAACGCGATCATGGCCGACGACGAGGTGCTCGTGAACAGCTTCTCGCGTTCGCTCGAACTCGGCGCGTTGCCGACCGTGCATGCGGAGAACGGCGAGCTCGTGTTCCAGTTGCAGAAGGCGCTGCTCGCGCGCGGGATGACGGGGCCGGAGGCGCATCCGCTGTCGCGGCCGCCGGAGGTCGAGGGTGAGGCGGCGAATCGTGCGATCCGCATTGCGCAGGTGCTCGGCGTGCCGGTGTATATCGTGCATGTGTCCGCGAAGGACGCGGTCGATGCGATCACGAAGGCGCGCAGCGAAGGGCTGCGCGTGTTCGGCGAGGTGCTGCCGGGCCATCTGGTGATCGACGAGGCCGTCTATCGCGATCCGGACTGGACACGTGCGGCCGCGCACGTGATGAGCCCGCCGTTCCGCTCGG

In [105]:
all_orfs_3 = find_orfs_fasta(fasta_file="./data/dna2.fasta", frame=3)
longest_in_file = max(all_orfs_3, key=lambda x: x["length"], default=None)

longest_in_file

{'sequence_id': 'gi|142022655|gb|EQ086233.1|527',
 'start': 636,
 'end': 2456,
 'length': 1821,
 'orf': 'ATGAACAGCGGGGCGAGCAAGCCGCCGGCCGTCACGGGGTCCATCACGAGGGACAGCAGCGGAATGCCGATGATCGCGAATCCACCACCGAACGCGCCGCGCATGAACGCGATCACGAACACGCCGGCAAACGCGATCAGGATCGTGGCCAGCGTCAATTGCAGGCCCATCGCAGCAGGGGTCGCCATCACGACCTCCATGCCGGTTCGAATCGCGGCGTGGCGGACAGCCACGGAGCGGGTCGCACGCGCGGCATCGCCGCACGATGGATCCGGGTTGAACGCGTTGCACCCATGCTGCTTCTCCAATGAGGTACCGGGGCGATGCGGTACACCAACGCACCGCAGGCCGCATGGGCCGCACAAGCATTTCAGCCCCGGTACAATCGACTTGACGAAAGCAGAATGCACCGCCGTCTATCTCAGTGCAATTAAAACATTGACCTCGGTGCAATATTCATTGTTATCGGTGCAATCCATGTCGAATTCCGAATACCTGCAGTTGGCCGACGCGATCGCCGCCCAAATTGCCGACGGCACGCTCAGGCCGGGCGACCGCCTGCCTCCGCAGCGTCATTTCGCCGACCAGCATGCGATCGCCGCATCGACGGCGGGACGGGTTTACGCGGAACTGTTACGGCGCGGCCTTGTGGTCGGCGAAGTCGGCCGAGGCACTTTCGTGTCGGGTGAGACGCGACGCGGGGCCGCTGCGCCGGGCGAGCCGCGCGGCGTTCGGATCGATTTCGAGTTCAACTACCCGACCGTCCCGGCCCAGACCGCGTTGATCACCAGAAGCCTGCGCGGATTGCACCGACCTGCGGAGCTCGACGCCGCGTTACGCGAGGCGACGAGTACCGGGACCCCGGTCATCCGAAGCGTTGCCGCCGCGTATCTGG

In [112]:
def get_longest(fasta_file: str | Path="./data/dna2.fasta") -> dict:
    all_orfs_1 = find_orfs_fasta(fasta_file, frame=1)
    all_orfs_2 = find_orfs_fasta(fasta_file, frame=2)
    all_orfs_3 = find_orfs_fasta(fasta_file, frame=3)
    longest_in_file = max(all_orfs_1 + all_orfs_2 + all_orfs_3, key=lambda x: x["length"])
    return longest_in_file

In [113]:
get_longest()

{'sequence_id': 'gi|142022655|gb|EQ086233.1|45',
 'start': 385,
 'end': 2778,
 'length': 2394,
 'orf': 'ATGGAGAAACAGTCTCGCGTTACGCGCGACGGTCGCGGGAGAGTTCTATGCGGTCATCGCTGCCGCGGTCGCGATTGGACTGGTCATGACGTTCGTTCATTTCGACCCGATTCGAGCGCTCTACTGGAGCGCCGTCATCAATGGGATCACGGCAGTGCCCATCATGGTGGTGATGATGCTGATGGCGCAGAGCCGGCGCGTGATGGGCGAGTTCGCAATCAGAGGACCGCTTGCGTGGGGAGGGTGGCTCGCGACGCTCGCCATGGCGCTCGCGGCGGCCGGAATGCTGCTGCCGGGATGAGCCGGCAATCCGGATGGAGAATGCGCATGCCCGCGACGCACCGGCGACGCCTCGCCGGACGGCGGGCGTCGCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCGAGCGCTCCATCGACGACGGTGGCGGCCACGCCCCGGAATTCGACATGCCTGCATCCTCCGATACGGCGAACCGGCGGGCGTCATCAATCGCGCGCATCCAGCGCGGGCTGAAGCGCGGGCTCGGCCGGCGCTGCCGGTTCATGGCCGCCGTGGCGCGCGGCGGTGGAATGGCCGGGCCGGATCCTGAACCAGATCGCATACATCGCGGGCAGGAACACGAGCGTGAGGACCGTCCCGGCGAACGTGCCGCCGATCAGCGTGTACGCGAGCGTGCCCCAGAACACCGAATGCGTGAGCGGAATGAACGCGAGCACGGCCGCCATCGCGGTAAGAATCACCGGGCGCGCCCGCTGCACGGTCGCTTCGACGACCGCGTGGAACGGATCGAGTCCCGCGTGTTCGTTCTGGTGGATCTGGCCGATCAGGATCAGCGTGTTGCGCATCA

In [115]:
all_orfs = []
for f in [1, 2, 3]:
    all_orfs = all_orfs + find_orfs_fasta(fasta_file="./data/dna2.fasta", frame=f)

In [116]:
all_orfs

[{'sequence_id': 'gi|142022655|gb|EQ086233.1|91',
  'start': 229,
  'end': 906,
  'length': 678,
  'orf': 'ATGCCGGCTTTCGCGATCGGCGCGAACACGCCGGCCGGCCTGCTCGCGTGGGGCTTGCCGGCGAATGCGTCGGCGGGCGGTGCGCTCGACAACCGCGTGTGGGGCGTCCAGGTGAACAATGCGGTGAAGTACGTGAGCCCGACGTTCGGCGGATTGTCGTTCGGCGGCCTGTGGGGCTTCGGCAACGTGCCCGGCACGGTCGCGCGCAGCAGCGTGCAAAGCGCGATGCTGTCCTACACGCAAGGCGCGTTCAGCGCCGCGCTCGCTTATTTCGGCCAGCACGATGTAACTGCCGGTGGCAATCTGCGCAATTTCTCGGGCGGTGCAGGCTACAACGTCGGGCAGTTCCGCGTCTTCGGCATGGTGTCGGACGTGCGGATCAGCGCCGCCGCGCCGCTGCGGGCCACGACCTATGACGGCGGCTTGACCTATGCGGTCACGCCGGCGTTGCAGCTCGGCGGCGGCTTCCAGTACCAGCAGCGCGGCGGCGACATCGGCTCGGCCAACCAGGTCACGTTGAGCGCCGACTATTCGCTGTCGAAGCGTACCGGCCTTTACGTGGTATTCGCACGCGGGCACGACAGTGCGTATGGCGCGCAGGTCGAGGCGGCGCTCGGCGGGGCGGCGTCCGGCTCGACGCAGACCGCGGTCCGGCTCGGGCTGCGGCATCAGTTCTGA',
  'frame': 1},
 {'sequence_id': 'gi|142022655|gb|EQ086233.1|91',
  'start': 454,
  'end': 906,
  'length': 453,
  'orf': 'ATGCTGTCCTACACGCAAGGCGCGTTCAGCGCCGCGCTCGCTTATTTCGGCCAGCACGATGTAACTGCCGGTGGCAATCTGCGCAATTTC

In [121]:
identifier = "gi|142022655|gb|EQ086233.1|16"

In [122]:
all_orfs_identified = [orf for orf in all_orfs if orf["sequence_id"] == identifier]

all_orfs_identified

[{'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
  'start': 265,
  'end': 330,
  'length': 66,
  'orf': 'ATGTCGTCAACGTCAGTTCGCGCTATGGCGCGGTGCAGTGGAACGGCCAGCGCATCGCGGGGCTGA',
  'frame': 1},
 {'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
  'start': 289,
  'end': 330,
  'length': 42,
  'orf': 'ATGGCGCGGTGCAGTGGAACGGCCAGCGCATCGCGGGGCTGA',
  'frame': 1},
 {'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
  'start': 577,
  'end': 603,
  'length': 27,
  'orf': 'ATGTGGAGATGGTCACGCGCTGGGTGA',
  'frame': 1},
 {'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
  'start': 1273,
  'end': 1494,
  'length': 222,
  'orf': 'ATGTGTGTCCGGTCGAGCAATGCATCACGATGGAGCGTGTCGATTCGGGCGACTACGCGAACTGGACCACGCATCCGAACAATCCGGCGAGCGCGGAGGCGGGGGCGAGTGCAGGCGCGGCGGCACCCGAGAAGCACGCGAAGAAGGCTGCTTGACGGCGTCCGGCGATGCGGGCCATCCTGCATCGCCGCCTTTCGTTCCACCCGGGCCGGCATCGAGTGA',
  'frame': 1},
 {'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
  'start': 1528,
  'end': 3036,
  'length': 1509,
  'orf': 'ATGAATCACGCAGCGAATCCCGCCGATCCCGATC

In [123]:
longest_in_file = max(all_orfs_identified, key=lambda x: x["length"])

longest_in_file

{'sequence_id': 'gi|142022655|gb|EQ086233.1|16',
 'start': 1440,
 'end': 3083,
 'length': 1644,
 'orf': 'ATGCGGGCCATCCTGCATCGCCGCCTTTCGTTCCACCCGGGCCGGCATCGAGTGATGCCGGCGTTGACGTTTTCGTGGAGTGAGTCAGATGAATCACGCAGCGAATCCCGCCGATCCCGATCGCGCCGCGGCGCAGGGCGGCAGCCTGTACAACGACGATCTCGCGCCGACGACGCCGGCGCAGCGCACGTGGAAGTGGTATCACTTCGCGGCGCTGTGGGTCGGGATGGTGATGAACATCGCGTCGTACATGCTCGCGGCCGGGCTGATCCAGGAAGGCATGTCGCCGTGGCAGGCGGTGACGACGGTGCTGCTCGGCAACCTGATCGTGCTCGTGCCGATGCTGCTGATCGGCCATGCGGGCGCGAAGCACGGGATTCCGTACGCGGTGCTCGTGCGCGCGTCGTTCGGCACGCAGGGGGCGAAGCTGCCGGCGCTGCTGCGCGCGATCGTCGCGTGCGGCTGGTACGGGATCCAGACCTGGCTCGGCGGCAGCGCGATCTATACGCTGCTGAACATCCTGACCGGCAACGCGCTGCATGGCGCCGCGCTGCCGGTCATCGGCATCGGGTTCGGGCAGCTCGCATGCTTCCTCGTGTTCTGGGCGCTGCAGCTCTACTTCATCTGGCATGGCACCGATTCGATCCGCTGGCTCGAAAGCTGGTCGGCGCCGATCAAGGTCGTGATGTGCGTGGCGCTGGTGTGGTGGGCAACGTCGAAGGCGGGCGGCTTCGGCACGATGCTGTCGGCGCCGTCGCAGTTTGCCGCAGGCGGCAAGAAAGCCGGGCTGTTCTGGGCGACCTTCTGGCCGGGGCTGACCGCGATGGTCGGCTTCTGGGCGACGCTCGCGCTGAACATCCCCGACTTCACGCGCTTCGCGCATTCGCAGCGCGAC

In [125]:
def find_repeats_fasta(fasta_file: str | Path, n: int):
    """
    Finds all repeated substrings of length n across all sequences in a FASTA file.

    Returns:
        repeats: dict of repeat -> count
        positions: dict of repeat -> list of (sequence_id, start_position)
        most_frequent: tuple of (repeat, count)
    """
    counts = Counter()
    positions = defaultdict(list)

    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_id = record.id
        seq = str(record.seq).upper()

        for i in range(len(seq) - n + 1):
            repeat = seq[i:i+n]
            counts[repeat] += 1
            positions[repeat].append((seq_id, i + 1))  # 1-based position

    repeats = {
        repeat: count
        for repeat, count in counts.items()
        if count > 1
    }

    repeat_positions = {
        repeat: positions[repeat]
        for repeat in repeats
    }

    most_frequent = max(repeats.items(), key=lambda x: x[1], default=None)

    return repeats, repeat_positions, most_frequent

In [130]:
find_repeats_fasta(fasta_file="./data/dna2.fasta", n=6)[2]

('GCGCGC', 153)

In [132]:
find_repeats_fasta(fasta_file="./data/dna2.fasta", n=12)[2]

('CATTCGCCATTC', 10)

In [134]:
find_repeats_fasta(fasta_file="./data/dna2.fasta", n=7)[2]

('CGCGCCG', 63)